# EWMA Crossover Strategy Research

**Research question:** Can an EWMA crossover strategy generate risk-adjusted returns?

- **H0:** EWMA crossover does not outperform random trading.
- **H1:** EWMA crossover generates positive risk-adjusted returns.

This notebook walks through the full pipeline: data → EWMA → signals → backtest → metrics → charts → optimization → **out-of-sample validation** → ML regime forecasting.

> **Methodology notes (read before trusting any single number below):**
> - Backtests use a **realistic next-open execution model** (signal known at `Close[t]` fills at `Open[t+1]`), not the unrealistic "trade at the same close the signal came from" assumption.
> - Sharpe ratios use an **approximate historical risk-free rate by year**, not a flat 0%.
> - The random-trading H0 test is run as a **bootstrap distribution across many resampled histories**, not just the one path we happened to load — a single-path percentile has real sampling noise.
> - The parameter grid search is followed by a **walk-forward validation** that tests chosen parameters on data they were never fit to.
> - The ML regime classifier is labeled from **strictly future returns**, never the same window used as its input features, and evaluated with walk-forward folds — no label leakage.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join("..", "src"))

import pandas as pd
import matplotlib.pyplot as plt

from data_loader import load_price_data
from ewma import build_feature_frame
from signals import generate_ewma_crossover_signals, trade_log
from backtester import run_backtest, run_random_baseline, compare_execution_models
from metrics import summary, summary_table, cagr, excess_cagr, sharpe_ratio, max_drawdown, annualized_volatility, historical_risk_free_rate
from visualize import plot_price_and_ewma, plot_signals, plot_portfolio_growth, plot_drawdown
from optimizer import grid_search
from validation import (
    walk_forward_strategy_validation, bootstrap_hypothesis_test,
    cost_sensitivity_analysis, hyperparameter_robustness_check,
)
from regime_model import (
    train_regime_classifier, build_regime_features, predict_current_regime,
    adaptive_ewma_params, backtest_regime_adaptive_strategy,
)
from multi_asset import run_cross_sectional_validation, run_scenario_validation, DEFAULT_BASKET

TICKER = "AAPL"
START, END = "2015-01-01", "2025-01-01"
FAST, SLOW = 20, 50
CAPITAL = 10_000


## Step 1–2: Load & clean data

In [ ]:
prices = load_price_data(TICKER, start=START, end=END)
prices = prices.dropna().sort_index()
prices.head()

## Step 3: EWMA engine

In [ ]:
feats = build_feature_frame(prices, fast_span=FAST, slow_span=SLOW)
feats[["Close", "EWMA_fast", "EWMA_slow", "returns"]].tail()

In [ ]:
plot_price_and_ewma(feats, title=f"{TICKER}: Price & EWMA")
plt.show()

## Step 4: Signal generation

In [ ]:
signaled = generate_ewma_crossover_signals(feats, flat_on_sell=True)
plot_signals(signaled, title=f"{TICKER}: Buy/Sell Signals")
plt.show()

## Step 5: Backtest

In [ ]:
bt = run_backtest(signaled, starting_capital=CAPITAL)  # execution_model="next_open" by default
trades = trade_log(signaled)
plot_portfolio_growth(bt)
plt.show()
plot_drawdown(bt)
plt.show()

### Execution realism: naive (same-close) vs. realistic (next-open)

`run_backtest()` defaults to a **realistic** execution model: a signal computed from `Close[t]` can only be known after the market closes that day, so the earliest price it can actually be traded at is the *next* session's `Open`. The position is then held until the following open, so its return is measured open-to-open, two bars after the signal that decided it.

The **naive** model (kept only for comparison) assumes you can trade at the exact same close price the signal was computed from — not achievable in real trading. The cell below shows what that unrealistic assumption was actually worth.

In [ ]:
exec_comparison = compare_execution_models(signaled, starting_capital=CAPITAL)
exec_comparison

## Step 6: Performance metrics

In [ ]:
strat_stats = summary(bt, trades)
bh_stats = {
    "CAGR": cagr(bt["buy_hold_value"]),
    "ExcessCAGR": excess_cagr(bt["buy_hold_value"]),
    "Sharpe": sharpe_ratio(bt["returns"]),
    "MaxDrawdown": max_drawdown(bt["buy_hold_value"]),
    "AnnualVolatility": annualized_volatility(bt["returns"]),
    "FinalValue": bt["buy_hold_value"].iloc[-1],
}
summary_table(("EWMA Strategy", strat_stats), ("Buy & Hold", bh_stats))

**Note on Sharpe ratios:** these use an approximate historical annualized risk-free rate by calendar year (see `metrics.historical_risk_free_rate`), not a flat 0%. 2015–2025 spanned roughly 0% to roughly 5% short-term rates, and using 0% throughout would have inflated every Sharpe ratio in this notebook.

In [ ]:
historical_risk_free_rate(bt.index).groupby(bt.index.year).first()

## Hypothesis test: EWMA vs. random trading (H0)

A single comparison against random traders on the one historical path we loaded would tell us very little about how *reliable* that result is — market history is one draw from a much larger space of paths that could have happened. To quantify the sampling noise, we **block-bootstrap resample** the historical returns many times (preserving short-run autocorrelation/volatility clustering), **recompute the EWMA signal fresh on each resampled path**, and record where the strategy lands vs. a fresh batch of random traders each time. The spread across draws is the sampling noise a single-run result would hide.

In [ ]:
boot = bootstrap_hypothesis_test(
    prices, fast_span=FAST, slow_span=SLOW, starting_capital=CAPITAL,
    n_bootstrap=100, n_random_traders=200, flat_on_sell=True,
)
pct = boot["percentile_vs_random"]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pct, bins=25, color="lightsteelblue", edgecolor="steelblue")
ax.axvline(50, color="gray", linestyle="--", linewidth=1, label="50th percentile (coin flip vs. random)")
ax.axvline(pct.mean(), color="tab:blue", linewidth=2, label=f"Mean = {pct.mean():.0f}th percentile")
ax.set_xlabel("Percentile vs. random traders, per bootstrap draw")
ax.set_ylabel("Count")
ax.set_title(f"Distribution of Outcomes Across {len(pct)} Bootstrap Resamples")
ax.legend()
plt.show()

print(pct.describe())
print(f"\nBeat the median random trader in {(pct > 50).mean():.0%} of resamples.")

**How to read this:** if the histogram is wide and straddles the 50th-percentile line, a single-run result (e.g. "beat 34% of random traders") is essentially noise — a different historical draw could easily have shown 50% or 60%. A narrow histogram clearly above (or below) 50 would be much stronger evidence for (or against) H1.

## Step 8: Parameter grid search (in-sample)

In [ ]:
results = grid_search(prices, fast_spans=[5, 10, 20, 30, 50], slow_spans=[50, 100, 200])
results.head(10)

⚠️ **This ranks parameter combinations on the same data used to evaluate them — textbook overfitting setup.** Picking the top row here and reporting only that number is exactly the mistake walk-forward validation exists to catch. See the next section.

## Walk-forward out-of-sample validation

For each fold: pick the best (fast, slow) combo using **only a training window** (via the same grid search above), then backtest that exact, already-fixed combo on a **later window it never saw**. Comparing the in-sample and out-of-sample columns directly shows whether the parameter search actually generalizes, or whether it's just fitting the past.

In [ ]:
wf = walk_forward_strategy_validation(prices, n_splits=5, gap=100, starting_capital=CAPITAL, flat_on_sell=True)
wf

In [ ]:
if not wf.empty:
    mean_is, mean_oos = wf["in_sample_Sharpe"].mean(), wf["oos_Sharpe"].mean()
    print(f"Mean in-sample Sharpe (what the search reported):  {mean_is:.3f}")
    print(f"Mean out-of-sample Sharpe (what actually happened): {mean_oos:.3f}")
    print(f"Gap: {mean_oos - mean_is:+.3f}")
    if mean_oos < mean_is - 0.15:
        print("\n=> Out-of-sample Sharpe is meaningfully lower than in-sample: the parameter")
        print("   search is likely overfitting to the training windows, not finding a real edge.")

### Does compound tuning overfit even more? And are these hyperparameters themselves robust?

Two things the walk-forward check above doesn't cover on its own:

1. It only tuned (fast, slow). A real user experimenting with this platform would also try different stop-loss and vol-target settings against the same historical data — tuning multiple axes together (compound overfitting) can find a spuriously good combination that no single-axis safeguard catches.
2. The walk-forward `gap`/`n_splits` and the bootstrap `block_size` were never validated choices — they're reasonable defaults. If the headline conclusions change a lot across nearby choices, that's worth knowing.

In [ ]:
wf_joint = walk_forward_strategy_validation(
    prices, n_splits=4, gap=100,
    fast_spans=(10, 20, 30), slow_spans=(50, 100),
    stop_loss_grid=[None, 0.08], target_vol_grid=[None, 0.15],
    starting_capital=CAPITAL,
)
print(wf_joint[["fold", "chosen_fast", "chosen_slow", "chosen_stop_loss", "chosen_target_vol",
                 "in_sample_Sharpe", "oos_Sharpe"]])

if not wf_joint.empty:
    print(f"\nSingle-axis search:  in-sample Sharpe {wf['in_sample_Sharpe'].mean():.3f} vs OOS {wf['oos_Sharpe'].mean():.3f}"
          f"  (gap: {wf['oos_Sharpe'].mean() - wf['in_sample_Sharpe'].mean():+.3f})")
    print(f"Compound search:     in-sample Sharpe {wf_joint['in_sample_Sharpe'].mean():.3f} vs OOS {wf_joint['oos_Sharpe'].mean():.3f}"
          f"  (gap: {wf_joint['oos_Sharpe'].mean() - wf_joint['in_sample_Sharpe'].mean():+.3f})")
    print("\nA bigger negative gap for the compound search means tuning multiple axes together overfits more.")

In [ ]:
robustness = hyperparameter_robustness_check(
    prices, gap_options=(100, 200), n_splits_options=(3, 5),
    block_size_options=(10, 20, 40), n_bootstrap=30,
)
print("Walk-forward OOS Sharpe across gap/n_splits choices (is the conclusion stable?):")
print(robustness["walk_forward_grid"])
print("\nBootstrap percentile across block_size choices (is the conclusion stable?):")
print(robustness["bootstrap_grid"])

**How to read this:** if `mean_oos_Sharpe` and `mean_percentile` stay in a similar ballpark across the different gap/n_splits/block_size rows, the earlier conclusions are robust to those specific choices. If they swing wildly, the earlier headline numbers were more fragile than a single run would suggest -- which is itself useful to know before trusting them.

## Step 9: ML market regime forecaster

Predicts the regime (Bull / Bear / HighVol / LowVol) over the **next 20 trading days** from today's trailing features (volatility, volume z-score, RSI, trend).

**Why "forecaster" and not "classifier" now:** an earlier version of this model labeled each row using the *same* trailing window it was also given as an input feature — the model wasn't predicting anything, it was reconstructing a threshold rule from data handed to it for free (hence the old ~100% accuracy, which was a red flag, not a result). Labels here are built from returns **strictly after** the feature date, and accuracy is measured with walk-forward folds plus a purge gap between train and test, so what's reported below is genuine out-of-sample forecasting skill — expect it to be modest, because regime forecasting is genuinely hard.

In [ ]:
model, cols, report = train_regime_classifier(prices)
print(report)

In [ ]:
regime_feats = build_regime_features(prices).dropna(subset=cols)
current_regime = predict_current_regime(model, cols, regime_feats.iloc[-1])
print(f"Forecast regime for the next 20 trading days: {current_regime}")
print(f"Suggested EWMA params: {adaptive_ewma_params(current_regime)}")

### Does actually USING the regime forecast help? (real backtest, not just a display)

The cell above shows a forecast and a "suggested" parameter switch — but a forecast alone doesn't tell you whether ACTING on it would have helped. This backtests a strategy that switches EWMA parameters based on the walk-forward-trained model's out-of-sample forecasts, chunk by chunk, and compares it to a static (fixed 20/50) baseline over the identical out-of-sample periods.

In [ ]:
adaptive_result = backtest_regime_adaptive_strategy(prices, starting_capital=CAPITAL, flat_on_sell=True)

if adaptive_result["adaptive_summary"]:
    comp = pd.DataFrame({
        "Adaptive (regime-switching)": adaptive_result["adaptive_summary"],
        "Static (fixed 20/50)": adaptive_result["static_summary"],
    }).T
    display_cols = [c for c in ["CAGR", "Sharpe", "MaxDrawdown", "FinalValue"] if c in comp.columns]
    print(comp[display_cols])

    adaptive_sharpe = adaptive_result["adaptive_summary"]["Sharpe"]
    static_sharpe = adaptive_result["static_summary"]["Sharpe"]
    if adaptive_sharpe > static_sharpe:
        print(f"\nAdaptive beat static by {adaptive_sharpe - static_sharpe:+.2f} Sharpe out-of-sample in this run.")
    else:
        print(f"\nAdaptive did NOT beat static ({adaptive_sharpe:.2f} vs {static_sharpe:.2f} Sharpe) -- "
              f"consistent with the classifier's accuracy being close to its majority-class baseline.")
else:
    print("Not enough data for a walk-forward adaptive backtest with the current date range.")

## Risk management: costs, sizing, and stop-losses

The backtests above used zero transaction costs and always-100%-in-or-out sizing — both silently optimistic. This section adds three things a real trading account can't avoid:

1. **Dynamic transaction costs** that scale with volatility and trade size vs. average dollar volume, instead of one flat bps number.
2. **Volatility-target position sizing** — scale exposure down in choppy markets, up (capped) in calm ones — instead of always betting 100%.
3. **A stop-loss overlay** — exit early if price moves too far against the entry, instead of waiting for the next crossover no matter how bad it gets.

In [ ]:
risk_variants = {
    "Baseline (no frictions, full size)": run_backtest(signaled, starting_capital=CAPITAL),
    "Flat 10bps cost": run_backtest(signaled, starting_capital=CAPITAL, transaction_cost_bps=10),
    "Dynamic cost model": run_backtest(signaled, starting_capital=CAPITAL, use_dynamic_costs=True),
    "Vol-target sizing (15%)": run_backtest(signaled, starting_capital=CAPITAL, target_vol=0.15),
    "Stop-loss (8%)": run_backtest(signaled, starting_capital=CAPITAL, stop_loss_pct=0.08),
    "All combined": run_backtest(signaled, starting_capital=CAPITAL, use_dynamic_costs=True, target_vol=0.15, stop_loss_pct=0.08),
}
risk_summary = pd.DataFrame({name: summary(bt_v) for name, bt_v in risk_variants.items()}).T
risk_summary[["CAGR", "Sharpe", "MaxDrawdown", "FinalValue"]]

## Transaction cost sensitivity: does trading frequency amplify cost drag?

The "28 trades over 10 years" example elsewhere in this project is survivable cost-wise almost by construction — low frequency means costs barely matter. A higher-frequency variant (smaller fast span, more crossovers) is a different story. This backtests several fast spans across a range of flat cost levels to show the degradation curve directly.

In [ ]:
cost_sens = cost_sensitivity_analysis(
    prices, fast_spans=(5, 10, 20, 30, 50), slow_span=100,
    cost_bps_grid=(0, 5, 10, 20, 40, 80), starting_capital=CAPITAL, flat_on_sell=True,
    n_periods=3,  # split into 3 sub-periods to check the degradation pattern isn't period-dependent
)
agg = cost_sens.groupby(["fast_span", "cost_bps"], as_index=False).agg(
    Sharpe=("Sharpe", "mean"), num_trades=("num_trades", "sum")
)
pivot_sharpe = agg.pivot(index="fast_span", columns="cost_bps", values="Sharpe")

fig, ax = plt.subplots(figsize=(9, 5))
for fast_span_val in pivot_sharpe.index:
    trades_n = agg[agg["fast_span"] == fast_span_val]["num_trades"].iloc[0]
    ax.plot(pivot_sharpe.columns, pivot_sharpe.loc[fast_span_val], marker="o",
            label=f"fast={fast_span_val} ({int(trades_n)} total trades)")
ax.set_xlabel("Transaction cost (bps per trade)")
ax.set_ylabel("Sharpe ratio (mean across 3 sub-periods)")
ax.set_title("Sharpe Degradation vs. Cost, by Trading Frequency")
ax.axhline(0, color="gray", linewidth=0.8)
ax.legend()
ax.grid(alpha=0.3)
plt.show()

pivot_sharpe

**How to read this:** compare how much each line drops from cost=0 to cost=80bps. The smaller-fast-span (higher-frequency, more trades) lines should drop noticeably faster than the larger-fast-span (lower-frequency) lines — the same strategy family can go from "marginal" to "clearly broken" purely from costs, depending on how often it trades.

## Does this generalize? Cross-sectional and cross-regime validation

Everything above is one ticker (AAPL) over one historical window (2015-2025). A strategy can look good (or bad) on that combination purely because of that decade's specific market character, and say nothing about a different sector, a bear-market decade, or a different asset class.

**Cross-sectional:** run the identical strategy on a basket of different tickers.

⚠️ **Survivorship bias warning:** `DEFAULT_BASKET` is a hard-coded, CURRENT list of tickers. This is a real methodological limitation, not just a caveat — companies that were delisted, went bankrupt, or were acquired during the backtest period are invisible from a list built today, which makes cross-sectional results look systematically better than a point-in-time-correct universe would. `run_cross_sectional_validation()` raises a `UserWarning` every time it's called with the default basket for exactly this reason (caught and shown below). There is no good fix for this without a licensed point-in-time constituents feed.

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    cross = run_cross_sectional_validation(start=START, end=END, fast_span=FAST, slow_span=SLOW, starting_capital=CAPITAL)
    for w in caught:
        print(f"UserWarning: {str(w.message)[:200]}...\n")

cross[[c for c in ["CAGR", "Sharpe", "MaxDrawdown", "NumTrades"] if c in cross.columns]]

**Cross-regime:** run the identical strategy against several synthetic market regimes with deliberately different drift/volatility characteristics (bull, bear, choppy sideways, high-vol crypto-like, low-vol grind) — illustrative stress tests, not historical replays, since sourcing genuinely different real historical regimes (a real bear-market decade, a real crypto asset, etc.) isn't possible offline.

In [ ]:
scenarios = run_scenario_validation(start=START, end=END, fast_span=FAST, slow_span=SLOW, starting_capital=CAPITAL, n_seeds=10)

fig, ax = plt.subplots(figsize=(9, 4.5))
colors = ["tab:green" if s > 0 else "tab:red" for s in scenarios["Sharpe_mean"]]
ax.bar(scenarios.index, scenarios["Sharpe_mean"], yerr=scenarios["Sharpe_std"], color=colors, capsize=4, alpha=0.85)
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_ylabel("Sharpe ratio (mean ± std across 10 seeds)")
ax.set_title("Strategy Sharpe Ratio Across Synthetic Market Regimes (10 seeds each)")
plt.xticks(rotation=20, ha="right")
plt.show()

n_reliably_positive = (scenarios["Sharpe_mean"] - scenarios["Sharpe_std"] > 0).sum()
print(f"Mean Sharpe positive AND more than one std above zero in {n_reliably_positive} of {len(scenarios)} tested regimes.")
scenarios[["annual_drift", "annual_vol", "Sharpe_mean", "Sharpe_std", "pct_seeds_positive_Sharpe", "CAGR_mean"]]

**Note the error bars.** An earlier version of this check ran each scenario with a single random seed and reported one Sharpe number per regime — exactly the single-path fragility the bootstrap hypothesis test elsewhere in this notebook exists to catch. Averaging across 10 seeds per scenario, and showing the std, makes that fragility visible: a scenario whose bar's error range crosses zero could easily have shown the opposite sign with a different draw.

**How to read this:** a strategy that's only profitable in one or two of these regimes (commonly: only the bull-market one) is a bet on that regime continuing, not a genuine, regime-independent edge. Compare this to the walk-forward and bootstrap results earlier in the notebook — if all three point the same direction (in-sample-only outperformance that doesn't survive out-of-sample testing, randomized resampling, OR regime changes), that's strong, convergent evidence the strategy as currently specified doesn't have a real edge.

## Conclusion

Summarize your findings here, using the validated (not just in-sample) numbers above:

- Did the EWMA strategy outperform buy & hold and the bootstrap random-trading distribution on a risk-adjusted basis — and how wide was that bootstrap distribution?
- Did the walk-forward out-of-sample Sharpe hold up anywhere near the in-sample/grid-search Sharpe, or did it collapse?
- Did the regime forecaster beat chance level (25% for 4 classes) by a meaningful margin?

If the honest answer to any of these is "no" or "barely," that's a legitimate research finding, not a failure of the notebook — it's exactly what rigorous out-of-sample validation is supposed to surface before anyone risks capital on a plausible-looking in-sample equity curve.